# 02 — The ML Workflow: Data → Model → Loss → Optimizer → Evaluate

**Difficulty:** ⭐⭐ (Beginner–Intermediate)  
**Estimated time:** 2 hours  
**Week:** 5 — ML Fundamentals & Linear Regression

---

## Why does this matter for ML?

Most beginners treat ML as a black box: dump data in, get predictions out. Practitioners understand the **internal loop**: where you start, what each step does, and — crucially — which step to revisit when things go wrong.

Understanding the workflow is what separates someone who runs `model.fit()` from someone who knows *why* the model is failing and how to fix it.

---

## The Real-World Analogy: A Chef's Workflow

| Chef Step | ML Step | Why |
|---|---|---|
| Gather & prep ingredients | **Collect & clean data** | Bad ingredients → bad dish no matter how skilled the chef |
| Choose a recipe | **Choose a model** | The recipe defines the family of possible outputs |
| Taste the dish | **Compute the loss** | Quantifies exactly how wrong the current result is |
| Adjust seasoning | **Run the optimizer** | Tweak the parameters to reduce the error |
| Serve to a critic | **Evaluate on test set** | Unbiased assessment by someone who never saw the cooking |
| Read the review, improve | **Iterate the loop** | If the critic hated it, go back and improve |

> **Key insight:** ML is a **loop**, not a linear pipeline. You cook → taste → adjust → cook again, until the dish is good enough to serve.

---

## Notebook Outline

1. Setup & Imports  
2. Step 1 — Data: Load, Explore, Split (70/15/15)  
3. Visualise the Feature Matrix  
4. Step 2 — Model: Linear Hypothesis  
5. Step 3 — Loss: MSE by Hand  
6. Visualise the Loss Surface (2D contour + 3D)  
7. Step 4 — Optimizer: Manual Gradient Descent  
8. Visualise Gradient Descent Path  
9. Step 5 — Evaluate: Train / Val / Test MSE  
10. Learning Curves (Bias–Variance)  
11. The Full Workflow Diagram  
12. Self-Check Questions

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D   # enables 3D plotting

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

print('Imports OK. Python version confirmed.')

---
## Step 1: Data — Collect, Clean, Split

### Plain English

Data is the raw material. Without good data, no amount of clever modelling will save you. This step has three substeps:

1. **Collect** — get the dataset (here: California Housing from sklearn)  
2. **Clean** — handle missing values, outliers, incorrect types  
3. **Split** — divide into training, validation, and test sets so you can measure generalisation

### The 70 / 15 / 15 Split

```
Full dataset (100%)
├── Training set   (70%) ← model learns weights here
├── Validation set (15%) ← YOU tune hyperparameters here; model never trains on this
└── Test set       (15%) ← touched ONCE at the very end; gives unbiased final score
```

**Why separate validation from test?**  
Every time you peek at validation performance and adjust your model, you indirectly "fit" to that set. The test set is the last line of defence — a critic who has never tasted your food before.

In [ ]:
# ── Cell 2: Load Data and Three-Way Split ────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
X_full = housing.data       # feature matrix: 20,640 rows × 8 features
y_full = housing.target     # target vector: median house value in $100k units

# --- Step A: Split off the final test set (15% of all data) ---
# This is put in a drawer and NOT touched until the very last step.
X_temp, X_test, y_temp, y_test = train_test_split(
    X_full, y_full,
    test_size=0.15,
    random_state=42
)

# --- Step B: Split the remaining 85% into train (70%) and validation (15%) ---
# 15/85 ≈ 0.176 gives us approximately 15% of total data as validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.176,
    random_state=42
)

print('=== Dataset Split Sizes ===')
print(f'Full dataset : {len(X_full):>6,} rows (100%)')
print(f'Train        : {len(X_train):>6,} rows ({len(X_train)/len(X_full)*100:.1f}%)')
print(f'Validation   : {len(X_val):>6,} rows ({len(X_val)/len(X_full)*100:.1f}%)')
print(f'Test         : {len(X_test):>6,} rows ({len(X_test)/len(X_full)*100:.1f}%)')
print(f'\nFeatures     : {list(X_full.columns)}')
print(f'Target range : ${y_full.min()*100:.0f}k – ${y_full.max()*100:.0f}k')

In [ ]:
# ── Cell 3: Visualise the Feature Matrix X ───────────────────────────────────
# "Your data is a matrix" — making this concrete

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Left: Show the first 12 rows of X as a heatmap (colour = value) ---
X_sample = X_train.head(12).copy()
# Scale to [0,1] per column so all columns are visible (z-score would work too)
X_norm = (X_sample - X_sample.min()) / (X_sample.max() - X_sample.min())
ax = axes[0]
im = ax.imshow(X_norm.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(X_full.columns)))
ax.set_xticklabels(X_full.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(12))
ax.set_yticklabels([f'House {i+1}' for i in range(12)], fontsize=8)
plt.colorbar(im, ax=ax, label='Normalised value (0=min, 1=max)')
ax.set_title('Feature Matrix X\n(each row = one census block, each col = one feature)', fontsize=11)

# --- Right: Distribution of the target variable y ---
ax = axes[1]
ax.hist(y_train, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(y_train.median(), color='red', linestyle='--',
           linewidth=2, label=f'Median = ${y_train.median()*100:.0f}k')
ax.set_xlabel('Median House Value ($100k)')
ax.set_ylabel('Count (census blocks)')
ax.set_title('Target Variable y (Training Set)\n'
             'Right-skewed: most blocks < $300k, few > $400k', fontsize=11)
ax.legend(fontsize=10)

plt.suptitle('Step 1: Explore the Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Feature Correlation Heatmap ──────────────────────────────────────
# Before building a model, understand which features are related to each other
# and which are most correlated with the target y

# Combine features and target into one DataFrame for correlation analysis
train_df = X_train.copy()
train_df['MedHouseVal'] = y_train.values

corr_matrix = train_df.corr()   # Pearson correlation coefficients

fig, ax = plt.subplots(figsize=(10, 8))
# Mask the upper triangle to avoid redundancy (correlation matrix is symmetric)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',      # show correlation values in each cell
    cmap='coolwarm',             # blue = negative, red = positive
    center=0,                    # white = zero correlation
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Feature Correlation Matrix (Training Set)\n'
             'MedInc has the strongest positive correlation with house price', fontsize=12)
plt.tight_layout()
plt.show()

# Show the target correlations sorted
print('Correlation with MedHouseVal (target):')
print(corr_matrix['MedHouseVal'].sort_values(ascending=False).round(3))

---
## Step 2: Choose a Model (The Hypothesis)

### Plain English

A model is your **guess about the form of the relationship** between inputs and output. You are saying: *"I believe the truth lives somewhere in this family of functions — help me find which one fits best."*

### The Linear Model

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$$

- $\hat{y}$ — the model's prediction
- $x_1, x_2, \ldots, x_n$ — the input features (e.g., income, rooms, latitude)
- $\theta_0$ — the **intercept** (bias): what the model predicts when all features = 0
- $\theta_1, \ldots, \theta_n$ — the **weights** (coefficients): how much each feature matters

### The model does NOT know θ yet

Before training, θ values are either random or zero. **Training finds the θ that makes predictions closest to the true y.** That search process is Steps 3 and 4.

In [ ]:
# ── Cell 5: Step 2 — Model Setup ─────────────────────────────────────────────
# For simplicity, use ONE feature (MedInc) for the manual gradient descent demo.
# We will later use all 8 features with sklearn's LinearRegression.

# Extract a single feature: Median Income (strongest predictor)
X_1d_train = X_train[['MedInc']].values    # shape: (n_train, 1)
X_1d_val   = X_val[['MedInc']].values
X_1d_test  = X_test[['MedInc']].values
y_tr = y_train.values    # convert pandas Series to numpy array
y_vl = y_val.values
y_te = y_test.values

# The hypothesis function for one feature:
# y_hat = theta0 + theta1 * MedInc
def linear_predict(X, theta0, theta1):
    """Compute y_hat = theta0 + theta1 * X for a 1-feature model."""
    return theta0 + theta1 * X.flatten()  # flatten: (n,1) → (n,)

# Visualise the data and a few candidate lines
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_1d_train[:, 0], y_tr, alpha=0.15, s=6, color='steelblue', label='Training data')

# Plot three "before training" guesses to motivate the need for optimisation
x_line = np.linspace(X_1d_train.min(), X_1d_train.max(), 200)
for (t0, t1, lbl, c) in [
    (0.0,  0.0,  'θ=(0, 0) — flat line',     'gray'),
    (0.5,  0.3,  'θ=(0.5, 0.3) — too flat',  'orange'),
    (0.45, 0.44, 'θ≈optimal — good fit',      'red'),
]:
    ax.plot(x_line, t0 + t1 * x_line, linewidth=2, label=lbl, color=c)

ax.set_xlabel('Median Income (MedInc)')
ax.set_ylabel('Median House Value ($100k)')
ax.set_title('Step 2: The Model is a Family of Lines\n'
             'Training finds the BEST θ₀, θ₁ for these data', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## Step 3: The Loss Function — Quantifying "How Wrong Am I?"

### Plain English

The loss function is a **single number** that tells you exactly how bad your current model is. If loss is high, the model is way off. If loss is near zero, the model is almost perfect.

The chef analogy: tasting the dish. Instead of vague feelings ("it's a bit bland"), the loss function gives you a precise score: "it needs 3.7g more salt."

### Mean Squared Error (MSE)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

- $(\hat{y}_i - y_i)$ — the **residual**: how far the prediction is from truth  
- We **square** it: (1) makes all errors positive, (2) penalises large errors disproportionately  
- We take the **mean** so the number doesn't grow with dataset size  

### The Loss Surface

If we sweep over all possible values of (θ₀, θ₁) and compute MSE at each point, we get a surface in 3D space — a **loss landscape**. Training is finding the lowest point on that landscape.

In [ ]:
# ── Cell 6: Step 3 — MSE Loss Function By Hand ───────────────────────────────

def mse_loss(y_true, y_pred):
    """Mean Squared Error: average of squared prediction errors."""
    errors = y_pred - y_true          # residuals: positive if over-predicted
    squared_errors = errors ** 2      # square: all positive, big errors hurt more
    return np.mean(squared_errors)    # average over all samples

# Demonstrate the loss for three different θ settings
theta_candidates = [
    (0.0,  0.0,  'No-op model (flat)'),
    (0.5,  0.3,  'Weak model'),
    (0.45, 0.44, 'Near-optimal model'),
]

print('=== Manual MSE Computation for Different θ ===')
print(f'{"Model":<25} {"θ₀":>6} {"θ₁":>6} {"Train MSE":>12}')
print('-' * 55)

for (t0, t1, label) in theta_candidates:
    y_hat = linear_predict(X_1d_train, t0, t1)
    loss  = mse_loss(y_tr, y_hat)
    print(f'{label:<25} {t0:>6.3f} {t1:>6.3f} {loss:>12.4f}')

print('\nObservation: lower MSE = better fit to the training data.')
print('The optimizer will automatically find the θ with the minimum MSE.')

In [ ]:
# ── Cell 7: Visualise the Loss Surface ───────────────────────────────────────
# Sweep θ₀ and θ₁ over a grid and compute MSE at every point
# This reveals the "bowl" shape of the MSE loss landscape

# Grid of candidate θ values
theta0_vals = np.linspace(-1.0, 2.0, 80)   # intercepts to test
theta1_vals = np.linspace(-0.1, 0.9, 80)   # slopes to test
T0, T1 = np.meshgrid(theta0_vals, theta1_vals)  # (80×80) grids

# Compute MSE for each (θ₀, θ₁) pair on the grid
MSE_grid = np.zeros_like(T0)
X_flat = X_1d_train.flatten()   # flatten once outside the loop for speed

for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        y_hat_ij = T0[i, j] + T1[i, j] * X_flat   # prediction for this (θ₀,θ₁)
        MSE_grid[i, j] = mse_loss(y_tr, y_hat_ij)

# Find the analytical minimum (for reference)
min_idx = np.unravel_index(MSE_grid.argmin(), MSE_grid.shape)
best_t0 = T0[min_idx]
best_t1 = T1[min_idx]
print(f'Grid minimum: θ₀={best_t0:.3f}, θ₁={best_t1:.3f}, MSE={MSE_grid.min():.4f}')

# --- Plot 1: 2D Contour (top-down view of the loss landscape) ---
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
contour = ax.contourf(T0, T1, MSE_grid, levels=40, cmap='viridis')
plt.colorbar(contour, ax=ax, label='MSE Loss')
ax.scatter(best_t0, best_t1, color='red', s=100, zorder=5,
           marker='*', label=f'Minimum MSE ({MSE_grid.min():.3f})')
ax.set_xlabel('θ₀ (intercept)')
ax.set_ylabel('θ₁ (slope for MedInc)')
ax.set_title('Loss Surface — Top-Down View\n'
             '(darker = lower MSE = better)', fontsize=12)
ax.legend(fontsize=9)

# --- Plot 2: 3D Surface ---
ax3d = fig.add_subplot(1, 2, 2, projection='3d', label='3d')
# Remove the existing 2D axes in slot [1] — we replaced it with 3D
axes[1].remove()

ax3d.plot_surface(T0, T1, MSE_grid, cmap='viridis', alpha=0.85, edgecolor='none')
ax3d.scatter([best_t0], [best_t1], [MSE_grid.min()],
             color='red', s=80, zorder=10, label='Minimum')
ax3d.set_xlabel('θ₀'); ax3d.set_ylabel('θ₁'); ax3d.set_zlabel('MSE')
ax3d.set_title('Loss Surface — 3D View\nGradient descent rolls\ndown this bowl', fontsize=10)
ax3d.view_init(elev=30, azim=-60)

plt.suptitle('Step 3: The MSE Loss Surface', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4: The Optimizer — Finding the Minimum

### Plain English

The optimizer's job is to **navigate the loss landscape** and find the θ values that sit at the bottom of the bowl. Gradient descent is the most fundamental optimizer.

### Gradient Descent — The Downhill Walk

Imagine you are standing blindfolded on a hilly landscape and want to reach the lowest valley. The strategy: feel which direction is downhill right where you stand, take a step in that direction, repeat.

Mathematically:

$$\theta \leftarrow \theta - \alpha \cdot \frac{\partial \mathcal{L}}{\partial \theta}$$

- $\alpha$ (alpha) — the **learning rate**: how big a step you take each iteration  
- $\frac{\partial \mathcal{L}}{\partial \theta}$ — the **gradient**: the direction of steepest *ascent* (we subtract it to go downhill)

### The Learning Rate Goldilocks Problem

| Learning Rate | Effect |
|---|---|
| Too small (0.0001) | Takes forever; might get stuck in local minima |
| Just right (0.01–0.1) | Smooth convergence to minimum |
| Too large (10+) | Overshoots the minimum; loss may INCREASE or oscillate |

### MSE Gradients for Linear Regression

$$\frac{\partial \text{MSE}}{\partial \theta_0} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)$$

$$\frac{\partial \text{MSE}}{\partial \theta_1} = \frac{2}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i) \cdot x_i$$

In [ ]:
# ── Cell 8: Step 4 — Manual Gradient Descent ─────────────────────────────────
# Implement gradient descent from scratch for a 1-feature linear model
# so we can watch the parameters update step-by-step

def compute_gradients(X_flat, y_true, theta0, theta1):
    """
    Compute gradients of MSE with respect to theta0 and theta1.
    Derivation: MSE = (1/n) sum((theta0 + theta1*x - y)^2)
    d/d_theta0 = (2/n) sum(y_hat - y)
    d/d_theta1 = (2/n) sum((y_hat - y) * x)
    """
    n = len(y_true)
    y_hat    = theta0 + theta1 * X_flat    # current predictions
    residuals = y_hat - y_true             # errors (predicted minus actual)
    grad_t0  = (2/n) * np.sum(residuals)           # gradient for intercept
    grad_t1  = (2/n) * np.sum(residuals * X_flat)  # gradient for slope
    return grad_t0, grad_t1

# ── Run gradient descent ──────────────────────────────────────────────────────
LEARNING_RATE = 0.05   # step size (alpha)
N_ITERATIONS  = 200    # how many update steps to take

# Start from a poor initial guess
theta0 = 0.0
theta1 = 0.0

X_tr_flat = X_1d_train.flatten()

# Record history for plotting
history = {'iteration': [], 'theta0': [], 'theta1': [], 'mse': []}

for iteration in range(N_ITERATIONS):
    # 1. Compute current loss
    y_hat = theta0 + theta1 * X_tr_flat
    loss  = mse_loss(y_tr, y_hat)
    
    # 2. Compute gradients — which direction to move
    grad_t0, grad_t1 = compute_gradients(X_tr_flat, y_tr, theta0, theta1)
    
    # 3. Update parameters — take a step downhill
    theta0 = theta0 - LEARNING_RATE * grad_t0
    theta1 = theta1 - LEARNING_RATE * grad_t1
    
    # Save snapshot every few steps (and always save the first 5)
    if iteration < 5 or iteration % 20 == 0:
        history['iteration'].append(iteration)
        history['theta0'].append(theta0)
        history['theta1'].append(theta1)
        history['mse'].append(loss)

print('=== Gradient Descent: First 5 Steps ===')
print(f'{"Step":>5} {"θ₀":>8} {"θ₁":>8} {"MSE":>10}')
print('-' * 38)
for i in range(min(5, len(history['iteration']))):
    print(f'{history["iteration"][i]:>5} '
          f'{history["theta0"][i]:>8.4f} '
          f'{history["theta1"][i]:>8.4f} '
          f'{history["mse"][i]:>10.4f}')
print(f'\nFinal θ₀={theta0:.4f}, θ₁={theta1:.4f}, MSE={mse_loss(y_tr, theta0+theta1*X_tr_flat):.4f}')

In [ ]:
# ── Cell 9: Visualise Gradient Descent Path ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: parameter path overlaid on the loss contour ---
ax = axes[0]
cf = ax.contourf(T0, T1, MSE_grid, levels=40, cmap='viridis', alpha=0.85)
plt.colorbar(cf, ax=ax, label='MSE')

# Draw the path taken by gradient descent
path_t0 = history['theta0']
path_t1 = history['theta1']
ax.plot(path_t0, path_t1, 'w-o', markersize=5, linewidth=1.5,
        label='GD path', zorder=5)
ax.scatter(path_t0[0],  path_t1[0],  color='lime',  s=100, zorder=6, label='Start')
ax.scatter(path_t0[-1], path_t1[-1], color='red',   s=100, zorder=6, label='End')
ax.set_xlabel('θ₀'); ax.set_ylabel('θ₁')
ax.set_title('Gradient Descent Path on Loss Surface\n'
             '(white line = parameter trajectory toward minimum)', fontsize=11)
ax.legend(fontsize=9)

# --- Right: MSE vs iteration (the "learning curve" of gradient descent) ---
ax = axes[1]
ax.plot(history['iteration'], history['mse'], 'o-',
        color='steelblue', linewidth=2, markersize=5)
ax.set_xlabel('Iteration (number of gradient steps)')
ax.set_ylabel('Training MSE')
ax.set_title('MSE Drops as Gradient Descent Runs\n'
             '(converges when the curve flattens)', fontsize=11)
ax.set_yscale('log')   # log scale reveals the initial fast drop clearly
plt.suptitle('Step 4: Gradient Descent — Rolling Down the Loss Bowl',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Learning Rate Comparison ────────────────────────────────────────
# Show what happens with three different learning rates: too small, right, too large

def run_gd(lr, n_iter=80):
    """Run gradient descent with given learning rate; return MSE history."""
    t0, t1 = 0.0, 0.0
    mse_history = []
    for _ in range(n_iter):
        y_hat = t0 + t1 * X_tr_flat
        mse_history.append(mse_loss(y_tr, y_hat))
        g0, g1 = compute_gradients(X_tr_flat, y_tr, t0, t1)
        t0 -= lr * g0
        t1 -= lr * g1
        # Clip to prevent numerical explosion for very large lr
        if abs(t0) > 1e6 or abs(t1) > 1e6:
            mse_history.extend([np.nan] * (n_iter - len(mse_history)))
            break
    return mse_history

lr_configs = [
    (0.001, 'Too small (α=0.001)', 'orange'),
    (0.05,  'Just right (α=0.05)', 'steelblue'),
    (0.45,  'Too large (α=0.45)',  'red'),
]

fig, ax = plt.subplots(figsize=(10, 6))
for (lr, label, color) in lr_configs:
    hist = run_gd(lr)
    ax.plot(hist, label=label, color=color, linewidth=2)

ax.set_xlabel('Iteration')
ax.set_ylabel('Training MSE (log scale)')
ax.set_yscale('log')
ax.set_ylim(bottom=0.1)   # clip very large values for legibility
ax.set_title('Learning Rate Goldilocks Problem\n'
             'Too small → slow. Too large → diverges. Just right → smooth convergence.',
             fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## Step 5: Evaluate — Train, Validation, and Test Performance

### Plain English

After training, we need to know: **does this model work on data it has never seen?** A model that memorises the training data perfectly (MSE ≈ 0 on training) but fails on new data is useless. This is called **overfitting**.

### The Three Numbers You Must Report

| Metric | What it measures | Analogy |
|---|---|---|
| **Training MSE** | Fit to data the model learned from | Grade on homework you already studied |
| **Validation MSE** | Generalisation — used for tuning | Grade on a practice exam |
| **Test MSE** | Final unbiased score | Grade on the real exam (only one chance!) |

### The Danger Signal: Overfitting

- **Training MSE ≪ Validation MSE** → the model memorised the training set  
- **Fix**: simplify the model, add regularisation (Ridge/Lasso), get more data

### Underfitting

- **Both MSEs are high** → the model is too simple to capture the pattern  
- **Fix**: more complex model (more features, polynomial terms, deeper network)

In [ ]:
# ── Cell 11: Step 5 — Evaluate All Three Splits ───────────────────────────────
# Now use all 8 features and sklearn's LinearRegression (uses the closed-form
# solution — the exact analytical minimum — which is faster than gradient descent
# for linear models)

# Scale all three splits using parameters fit ONLY on the training set
scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)   # fit + transform on training data
X_vl_sc  = scaler.transform(X_val)         # transform only (no fit)
X_te_sc  = scaler.transform(X_test)        # transform only (no fit)

# Train the model
model = LinearRegression()
model.fit(X_tr_sc, y_train)

# Compute MSE on all three splits
train_mse = mean_squared_error(y_train, model.predict(X_tr_sc))
val_mse   = mean_squared_error(y_val,   model.predict(X_vl_sc))
test_mse  = mean_squared_error(y_test,  model.predict(X_te_sc))

train_r2  = r2_score(y_train, model.predict(X_tr_sc))
val_r2    = r2_score(y_val,   model.predict(X_vl_sc))
test_r2   = r2_score(y_test,  model.predict(X_te_sc))

print('=== Step 5: Model Evaluation ===')
print(f'{"Split":<12} {"MSE":>8} {"R²":>8}')
print('-' * 30)
print(f'{"Train":<12} {train_mse:>8.4f} {train_r2:>8.4f}')
print(f'{"Validation":<12} {val_mse:>8.4f} {val_r2:>8.4f}')
print(f'{"Test":<12} {test_mse:>8.4f} {test_r2:>8.4f}')
print()
if val_mse / train_mse > 1.5:
    print('WARNING: Validation MSE is significantly higher than Train MSE.')
    print('This suggests overfitting — the model has memorised training data.')
else:
    print('Train and Validation MSEs are close — healthy generalisation.')

In [ ]:
# ── Cell 12: Learning Curves — Bias vs Variance (MSE vs Model Complexity) ────
# Train polynomial models of degree 1–8
# A degree-1 model is simple (high bias), degree-8 is complex (high variance)
# Plot Train MSE and Val MSE for each degree — the gap reveals over/underfitting

MAX_DEGREE = 8
degrees    = range(1, MAX_DEGREE + 1)
train_mses = []
val_mses   = []

# Use only one feature (MedInc) so polynomial features don't explode in count
X_1d_tr_vals = X_train[['MedInc']].values
X_1d_vl_vals = X_val[['MedInc']].values

for deg in degrees:
    # Pipeline: polynomial expand → scale → linear regression
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=deg, include_bias=False)),
        ('scale', StandardScaler()),
        ('reg',   LinearRegression())
    ])
    pipe.fit(X_1d_tr_vals, y_train)
    
    train_mses.append(mean_squared_error(y_train, pipe.predict(X_1d_tr_vals)))
    val_mses.append(  mean_squared_error(y_val,   pipe.predict(X_1d_vl_vals)))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(degrees, train_mses, 'o-', color='steelblue',  linewidth=2.5,
        markersize=8, label='Train MSE')
ax.plot(degrees, val_mses,   's--', color='darkorange', linewidth=2.5,
        markersize=8, label='Validation MSE')

# Shade regions for illustration
ax.axvspan(0.5, 2.5, alpha=0.08, color='blue',  label='Underfitting zone (high bias)')
ax.axvspan(4.5, 8.5, alpha=0.08, color='red',   label='Overfitting zone (high variance)')

ax.set_xlabel('Polynomial Degree (model complexity)')
ax.set_ylabel('MSE')
ax.set_title('Learning Curves: Train vs Validation MSE by Model Complexity\n'
             'Degree-1 = underfits. High degree = overfits. Sweet spot is in between.',
             fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(list(degrees))
plt.tight_layout()
plt.show()

# Print the sweet spot
best_deg = degrees[np.argmin(val_mses)]
print(f'Best polynomial degree (lowest val MSE): degree = {best_deg}')
print(f'Train MSE at degree {best_deg}: {train_mses[best_deg-1]:.4f}')
print(f'Val   MSE at degree {best_deg}: {val_mses[best_deg-1]:.4f}')

In [ ]:
# ── Cell 13: Prediction Quality on Test Set ───────────────────────────────────
# Final unbiased evaluation using the test set (touched for the first time)

y_test_pred = model.predict(X_te_sc)   # predictions from the full 8-feature model

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Actual vs Predicted on test set ---
ax = axes[0]
ax.scatter(y_te, y_test_pred, alpha=0.2, s=8, color='steelblue')
lims = [y_te.min(), y_te.max()]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect (y=x)')
ax.set_xlabel('Actual ($100k)')
ax.set_ylabel('Predicted ($100k)')
ax.set_title(f'Test Set: Actual vs Predicted\nR²={test_r2:.3f}, MSE={test_mse:.4f}', fontsize=12)
ax.legend(fontsize=9)

# --- Right: Feature importance (magnitude of standardised coefficients) ---
ax = axes[1]
coefs = pd.Series(model.coef_, index=X_full.columns)
coefs_sorted = coefs.abs().sort_values(ascending=True)
coefs_sorted.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_xlabel('Absolute Coefficient Magnitude\n(proxy for feature importance)')
ax.set_title('What the Model Learned:\nWhich Features Drive Price Predictions?', fontsize=12)
ax.axvline(0, color='black', linewidth=0.8)

plt.suptitle('Step 5: Final Test Set Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 14: The ML Workflow — Full Diagram ───────────────────────────────────
# Draw the iterative loop as a matplotlib diagram with boxes and arrows

fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16); ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#f0f4f8')
fig.patch.set_facecolor('#f0f4f8')

# Step boxes: (x, y, width, height, title, subtitle, colour)
steps = [
    (0.3,  3.5, 2.5, 2.2, 'STEP 1\nData',
     'Collect → Clean\nSplit 70/15/15\nScale features', '#1f77b4'),
    (3.2,  3.5, 2.5, 2.2, 'STEP 2\nModel',
     'Choose hypothesis\ny = θ₀+θ₁x₁+...\nSet initial θ', '#2ca02c'),
    (6.1,  3.5, 2.5, 2.2, 'STEP 3\nLoss',
     'Compute MSE\nHow wrong now?\nLoss surface', '#ff7f0e'),
    (9.0,  3.5, 2.5, 2.2, 'STEP 4\nOptimiser',
     'Gradient descent\nUpdate θ\nRepeat N times', '#9467bd'),
    (11.9, 3.5, 2.5, 2.2, 'STEP 5\nEvaluate',
     'Train/Val/Test\nMSE, R²\nLearning curves', '#d62728'),
]

for (bx, by, bw, bh, btitle, bsub, bc) in steps:
    rect = mpatches.FancyBboxPatch(
        (bx, by), bw, bh,
        boxstyle='round,pad=0.15',
        facecolor=bc, edgecolor='white', linewidth=2.5, alpha=0.92
    )
    ax.add_patch(rect)
    ax.text(bx + bw/2, by + bh - 0.45, btitle,
            ha='center', va='center', fontsize=12, fontweight='bold', color='white')
    ax.text(bx + bw/2, by + bh/2 - 0.2, bsub,
            ha='center', va='center', fontsize=8.5, color='#f0f0f0',
            linespacing=1.5)

# Forward arrows between boxes
arrow_props = dict(arrowstyle='->', color='#333', lw=2.0)
for start_x in [2.8, 5.7, 8.6, 11.5]:
    ax.annotate('', xy=(start_x + 0.4, 4.6), xytext=(start_x, 4.6),
                arrowprops=arrow_props)

# Feedback loop arrow: "Too high val loss → go back"
ax.annotate('',
    xy=(1.55, 3.5), xytext=(13.15, 3.5),
    arrowprops=dict(
        arrowstyle='->', color='#c00', lw=2.5,
        connectionstyle='arc3,rad=-0.5'
    )
)
ax.text(7.4, 1.55, 'Val loss too high? → Adjust model complexity or hyperparams → retrain',
        ha='center', va='center', fontsize=10, color='#c00',
        fontweight='bold')

# House price context labels below each box
context = [
    (1.55,  3.25, 'Cal. Housing\n70/15/15 split'),
    (4.45,  3.25, 'y = θ₀ + θ₁·Income\n+ ... + θ₈·Lat'),
    (7.35,  3.25, 'MSE on price\nresiduals'),
    (10.25, 3.25, 'α=0.05\n200 steps'),
    (13.15, 3.25, 'Test R²=0.60\nMSE=0.53'),
]
for (cx, cy, ctxt) in context:
    ax.text(cx, cy, ctxt, ha='center', va='center',
            fontsize=8, color='#444', style='italic', linespacing=1.4)

ax.set_title('The ML Workflow: A Loop, Not a Pipeline\n'
             '(California Housing — Predict Median House Price)',
             fontsize=15, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 15: Putting It All Together — Summary Statistics ─────────────────────
# Show the complete story: before training (baseline) vs after training

# Baseline: always predict the training mean (the simplest possible model)
baseline_pred_tr = np.full_like(y_tr, fill_value=y_train.mean())
baseline_pred_te = np.full_like(y_te, fill_value=y_train.mean())
baseline_mse_tr  = mse_loss(y_tr, baseline_pred_tr)
baseline_mse_te  = mse_loss(y_te, baseline_pred_te)

# Linear Regression (trained)
final_mse_tr = mean_squared_error(y_train, model.predict(X_tr_sc))
final_mse_te = mean_squared_error(y_test,  model.predict(X_te_sc))

print('=== Complete Workflow Summary ===')
print()
print('STEP 1 — Data')
print(f'  Samples: {len(X_full):,} total | {len(X_train):,} train '
      f'| {len(X_val):,} val | {len(X_test):,} test')
print(f'  Features: {X_full.shape[1]}')
print()
print('STEP 2 — Model')
print(f'  LinearRegression with {X_full.shape[1]} features')
print(f'  Parameters to learn: {X_full.shape[1]+1} (1 intercept + {X_full.shape[1]} slopes)')
print()
print('STEP 3 — Loss Function')
print(f'  MSE = (1/n) sum((y_hat - y)^2)')
print(f'  Baseline MSE (predict mean): {baseline_mse_tr:.4f} (train), {baseline_mse_te:.4f} (test)')
print()
print('STEP 4 — Optimizer')
print(f'  sklearn uses closed-form OLS: θ = (XᵀX)⁻¹ Xᵀy')
print(f'  Manual GD demo: α=0.05, 200 iterations')
print()
print('STEP 5 — Evaluate')
print(f'  Baseline → Train MSE: {baseline_mse_tr:.4f} | Test MSE: {baseline_mse_te:.4f}')
print(f'  Trained  → Train MSE: {final_mse_tr:.4f}   | Test MSE: {final_mse_te:.4f}')
print(f'  MSE improvement on test: {(1 - final_mse_te/baseline_mse_te)*100:.1f}% reduction')
print(f'  Test R²: {r2_score(y_test, model.predict(X_te_sc)):.4f}')

---
## The Workflow Loop — Annotated

```
┌─────────────────────────────────────────────────────────────────────┐
│                      THE ML WORKFLOW LOOP                           │
│                                                                     │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────────┐  │
│  │  DATA    │───►│  MODEL   │───►│  LOSS    │───►│  OPTIMIZER   │  │
│  │ Collect  │    │ y=θ₀+θx  │    │  MSE     │    │ Gradient     │  │
│  │ Clean    │    │ params θ │    │ surface  │    │ Descent      │  │
│  │ Split    │    │ init rand│    │ bowl     │    │ Update θ     │  │
│  └──────────┘    └──────────┘    └──────────┘    └──────┬───────┘  │
│                                                          │          │
│                                                          ▼          │
│                                                  ┌──────────────┐  │
│            ◄─────── Val loss too high? ──────────│  EVALUATE    │  │
│            adjust complexity / hyperparams        │ Train/Val/   │  │
│            or get more data                       │ Test MSE, R² │  │
│                                                  └──────┬───────┘  │
│                                                         │          │
│                                            Val loss OK? │          │
│                                                         ▼          │
│                                            ┌─────────────────────┐ │
│                                            │  REPORT TEST SCORE  │ │
│                                            │  (touch test set    │ │
│                                            │   ONCE, at the end) │ │
│                                            └─────────────────────┘ │
└─────────────────────────────────────────────────────────────────────┘
```

---

## When to Stop the Loop — Early Stopping

In iterative training (gradient descent, neural networks, gradient boosted trees):  

1. Track validation loss after every epoch (pass through the data)
2. If validation loss stops improving for N consecutive epochs → **stop training**  
3. Restore the weights from the epoch with the best validation loss

This prevents overfitting without needing to know the perfect number of epochs in advance.

---
## Self-Check Questions

Answer these before moving to the next notebook. Cover the answers with your hand first.

---

### Question 1
Your model's **training MSE is 0.08** but **validation MSE is 0.95**. What does this tell you, and what would you do next in the workflow loop?

<details>
<summary>Click to reveal answer</summary>

**Answer: This is a classic overfitting signature.**

The model has essentially memorised the training set (near-zero error) but completely fails to generalise — it has learned noise and quirks of the training data, not the underlying pattern.

**What to do:**
1. **Simplify the model** — reduce polynomial degree, reduce number of features, use a less expressive model class
2. **Add regularisation** — Ridge (L2) or Lasso (L1) regression penalises large weights and forces the model to be more general
3. **Get more training data** — more data makes it harder to overfit
4. **Feature selection** — remove irrelevant or redundant features
5. **Cross-validation** — use k-fold CV to get a more robust estimate of generalisation

In the workflow loop, you return to **Step 2 (Model)** and modify the hypothesis (simpler model), or to **Step 1 (Data)** to get more examples.

</details>

---

### Question 2
You increase the learning rate from 0.01 to 10. What might happen to the optimizer?

<details>
<summary>Click to reveal answer</summary>

**Answer: The optimizer will likely diverge.**

With a very large learning rate, each gradient step overshoots the minimum. Instead of rolling down into the bowl, the parameters bounce back and forth across the bowl, landing on the far wall, which has an even steeper gradient, causing an even bigger step in the opposite direction — and so on.

In practice you will see:
- Training loss **increasing** over iterations instead of decreasing
- Loss values becoming `NaN` or `inf` (numerical explosion)
- The parameter values growing without bound

**Fix:** Reduce the learning rate. A common starting point is α=0.01 or α=0.001. Many modern optimizers (Adam, RMSprop) adapt the learning rate automatically per parameter.

</details>

---

### Question 3
Why do we need a SEPARATE test set if we already have a validation set?

<details>
<summary>Click to reveal answer</summary>

**Answer: Because the validation set becomes "contaminated" through repeated use.**

Every time you:
- Look at validation loss and decide to increase/decrease model complexity
- Try a different regularisation strength because validation loss was high
- Tune learning rate based on validation performance
- Choose between two candidate models based on validation score

...you are making choices that are guided by the validation set. Even without the model ever training on validation data, YOUR decisions as a practitioner encode information from it. Over many iterations, the final model is optimised for the validation set.

The **test set** is the only data point the practitioner never made any decision based on. It provides a **truly unbiased estimate** of how the model will perform on completely new, real-world data.

**Analogy:** The validation set is the practice exam. The test set is the actual final exam you take once — and you cannot retake it after seeing your score.

</details>

---

### Question 4 (Bonus)
In the manual gradient descent demo, we used a learning rate of 0.05 and ran for 200 iterations. How would you know if more iterations were needed?

<details>
<summary>Click to reveal answer</summary>

**Answer:** Plot the **training loss vs iteration** curve (which we did in Cell 9).

If the curve is **still declining** at iteration 200, then more iterations are needed — training hasn't converged yet.

If the curve has **flattened** (loss barely changing between iterations 150–200), then the optimizer has converged and more iterations won't help.

Quantitatively: check if `|loss(t) - loss(t-1)| < ε` for some small tolerance ε (e.g., 1e-6). This is the **convergence criterion**. Most sklearn models handle this automatically via the `tol` parameter.

</details>

---

### Question 5 (Bonus)
Why do we scale features (StandardScaler) BEFORE fitting the model? What happens if we forget to scale?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

Feature scaling is critical for two reasons:

**1. Gradient descent convergence speed:**  
If one feature ranges from 0–1 and another from 0–100,000, the loss surface is elongated and stretched. Gradient descent will oscillate in the high-magnitude direction and barely move in the low-magnitude direction — making convergence very slow. After scaling, the loss surface is more spherical and gradient descent converges in far fewer steps.

**2. Model interpretability:**  
After scaling, all features are on the same scale. The magnitude of each learned coefficient (weight) directly reflects how much each feature *matters*. Without scaling, a large coefficient might just mean a small-scale feature, not an important one.

**Critical warning:** Always fit the scaler on the TRAINING data only, then apply (transform) to validation and test. If you fit on the full dataset first, you leak statistics from the test set into training — a subtle form of data leakage that inflates performance estimates.

</details>